# Basis and angle encoding
**Quantum Machine Learning · Week 1 (Data meets qubits) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/qml/w1d2.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

Before a quantum computer can learn from data, the data must become a quantum state. Two simple
ways: **basis encoding** for bits, and **angle encoding** for real numbers.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp, state_fidelity

def basis_encoding(bits):
    """Bit string -> computational basis state. The last character goes to qubit 0, as Qiskit prints."""
    qc = QuantumCircuit(len(bits))
    for q, b in enumerate(reversed(bits)):
        if b == "1":
            qc.x(q)
    return qc

state = Statevector(basis_encoding("1011"))
print("1011 encodes as:", state.probabilities_dict(), " (one basis state, with certainty)")
print("n bits need n qubits; the state holds exactly one bit string.")

## Angle encoding
Each real feature $x_j$ becomes the angle of an $R_Y$ rotation on its own qubit:
$|\varphi(x)\rangle = \bigotimes_j \big(\cos\frac{x_j}{2}|0\rangle + \sin\frac{x_j}{2}|1\rangle\big)$. Measuring $Z$ on
qubit $j$ gives back $\langle Z_j\rangle = \cos x_j$.

In [ ]:
def angle_encoding(x):
    qc = QuantumCircuit(len(x))
    for q, value in enumerate(x):
        qc.ry(value, q)
    return qc

x = np.array([0.3, 1.2, 2.5])
sv = Statevector(angle_encoding(x))
for j in range(len(x)):
    Z_j = SparsePauliOp.from_sparse_list([("Z", [j], 1.0)], num_qubits=len(x))
    print(f"qubit {j}: <Z> = {sv.expectation_value(Z_j).real:+.4f}, cos(x_{j}) = {np.cos(x[j]):+.4f}")

## Scale the features first
Angles repeat every $2\pi$, so raw features can collide: $x$ and $x + 2\pi$ give the same state. And tiny
ranges make every state look alike. Rescaling each feature to $[0, \pi]$ sweeps each qubit from $|0\rangle$
to $|1\rangle$ exactly once.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

collide = state_fidelity(Statevector(angle_encoding([0.5])), Statevector(angle_encoding([0.5 + 2 * np.pi])))
print(f"fidelity of x = 0.5 and x = 0.5 + 2*pi: {collide:.4f}  (the same state!)")

raw = np.array([[12.0, 0.003], [55.0, 0.009], [97.0, 0.006]])     # features on very different scales
scaled = MinMaxScaler((0, np.pi)).fit_transform(raw)
print("scaled to [0, pi]:\n", scaled.round(3))
for i in range(3):
    for k in range(i + 1, 3):
        F = state_fidelity(Statevector(angle_encoding(scaled[i])), Statevector(angle_encoding(scaled[k])))
        print(f"   points {i} and {k}: fidelity {F:.3f}")

## Your turn: dense angle encoding
Put **two** features on each qubit: $R_Y(x_1)$ then $R_Z(x_2)$ on qubit 0, $R_Y(x_3)$ then $R_Z(x_4)$ on qubit 1.
Four features, two qubits. Encode the four points below and check, with the fidelity, that no two of them
give the same state.

In [ ]:
def dense_encoding(x):
    qc = QuantumCircuit(len(x) // 2)
    # TODO: for each qubit q, ry(x[2q]) then rz(x[2q + 1])
    return qc

points = np.array([[0.4, 0.0, 1.0, 2.0], [0.4, 1.5, 1.0, 2.0], [2.0, 0.0, 1.0, 2.0], [0.4, 0.0, 1.0, 0.5]])
states = [Statevector(dense_encoding(p)) for p in points]
print([[round(state_fidelity(a, b), 3) for b in states] for a in states])

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def dense_encoding(x):
    qc = QuantumCircuit(len(x) // 2)
    for q in range(len(x) // 2):
        qc.ry(x[2 * q], q)                    # latitude on the Bloch sphere
        qc.rz(x[2 * q + 1], q)                # longitude
    return qc

states = [Statevector(dense_encoding(p)) for p in points]
F = np.array([[state_fidelity(a, b) for b in states] for a in states])
print(np.round(F, 3))
print("Only the diagonal is 1: four features, two qubits, four different states.")
print("Careful: a feature in RZ only matters if the RY angle isn't 0 or pi (at the poles, longitude means nothing).")